# Analisi Diffusione COVID-19

Progetto di analisi dati basato sul dataset **Our World in Data**.

**Dataset:** https://raw.githubusercontent.com/owid/covid-19-data/refs/heads/master/public/data/owid-covid-data.csv

---

In [ ]:
# Import delle librerie necessarie
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')

# Stile grafici
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3
sns.set_palette('tab10')

print('Librerie importate correttamente.')

---
## Punto 1 — Caricamento del dataset e EDA preliminare

In [ ]:
# Caricamento del dataset
URL = 'https://raw.githubusercontent.com/owid/covid-19-data/refs/heads/master/public/data/owid-covid-data.csv'

df = pd.read_csv(URL, low_memory=False)

print(f'Dataset caricato con successo.')
print(f'Dimensioni: {df.shape[0]:,} righe x {df.shape[1]} colonne')

In [ ]:
# Visualizzazione delle prime righe
df.head(3)

In [ ]:
# Metadati: tipi di dato e valori non nulli
df.info()

In [ ]:
# Statistiche descrittive delle colonne numeriche principali
cols_interesse = ['total_cases', 'new_cases', 'total_deaths', 'new_deaths',
                  'icu_patients', 'hosp_patients', 'total_vaccinations']
df[cols_interesse].describe()

In [ ]:
# ⚠️  NOTA IMPORTANTE — struttura delle colonne 'continent' e 'location'
#
# Il dataset contiene sia righe per singole nazioni (continent valorizzato)
# sia righe aggregate per continente/mondo (continent NaN, location = es. 'World', 'Europe', ...)
#
# Esempio di location aggregate:
aggregate_locations = df[df['continent'].isna()]['location'].unique()
print('Location aggregate (continent == NaN):')
print(aggregate_locations)

print()
print('Continenti presenti:')
print(df['continent'].dropna().unique())

In [ ]:
# ⚠️  NOTA — differenza tra new_cases e total_cases
#
# - new_cases: nuovi casi registrati nel periodo (giornaliero o settimanale a seconda della nazione)
#              → da SOMMARE per ottenere totali su un periodo
# - total_cases: valore cumulativo progressivo fino a quella data
#                → va preso come ULTIMO valore disponibile, NON sommato
#
# Sommare total_cases produrrebbe un valore enormemente sovrastimato!

# Conversione della colonna date in datetime
df['date'] = pd.to_datetime(df['date'])
print('Colonna date convertita in datetime.')
print(f"Intervallo temporale: {df['date'].min().date()} → {df['date'].max().date()}")

---
## Punto 2 — Casi totali per continente dall'inizio della pandemia

In [ ]:
# Filtro: solo righe con continente valorizzato (esclude aggregati tipo 'World', 'Europe', ...)
df_nazioni = df[df['continent'].notna()].copy()

# Per ogni nazione prendiamo il valore massimo di total_cases
# (= ultimo valore cumulativo disponibile, che rappresenta il totale storico)
casi_per_nazione = (
    df_nazioni
    .groupby(['continent', 'location'])['total_cases']
    .max()
    .reset_index()
)

# Somma per continente
casi_per_continente = (
    casi_per_nazione
    .groupby('continent')['total_cases']
    .sum()
    .reset_index()
    .rename(columns={'total_cases': 'casi_totali'})
    .sort_values('casi_totali', ascending=False)
)

# Totale mondiale (somma dei continenti)
totale_mondiale = casi_per_continente['casi_totali'].sum()

# Calcolo percentuale
casi_per_continente['percentuale_%'] = (
    (casi_per_continente['casi_totali'] / totale_mondiale * 100)
    .round(2)
)

casi_per_continente['casi_totali_fmt'] = casi_per_continente['casi_totali'].apply(lambda x: f"{x:,.0f}")

print(f"Totale mondiale stimato (somma continenti): {totale_mondiale:,.0f}")
print()
display(
    casi_per_continente[['continent', 'casi_totali_fmt', 'percentuale_%']]
    .rename(columns={'continent': 'Continente', 'casi_totali_fmt': 'Casi Totali', 'percentuale_%': '% Mondiale'})
    .reset_index(drop=True)
)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# --- Grafico 1: casi totali ---
axes[0].bar(
    casi_per_continente['continent'],
    casi_per_continente['casi_totali'] / 1e6,
    color=sns.color_palette('tab10', len(casi_per_continente))
)
axes[0].set_title('Casi totali per continente', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Continente')
axes[0].set_ylabel('Casi (milioni)')
axes[0].tick_params(axis='x', rotation=15)

# --- Grafico 2: torta percentuali ---
axes[1].pie(
    casi_per_continente['percentuale_%'],
    labels=casi_per_continente['continent'],
    autopct='%1.1f%%',
    startangle=140,
    colors=sns.color_palette('tab10', len(casi_per_continente))
)
axes[1].set_title('% casi per continente sul totale mondiale', fontsize=13, fontweight='bold')

plt.tight_layout()
plt.show()

---
## Punto 3 — Italia 2022: evoluzione casi totali e nuovi casi settimanali

In [ ]:
# Selezione dati Italia 2022
df_ita = df[
    (df['location'] == 'Italy') &
    (df['date'].dt.year == 2022)
].copy()

print(f'Righe Italia 2022 (prima del filtro): {len(df_ita)}')

# I nuovi casi vengono registrati settimanalmente → rimuoviamo le righe senza misurazione
# (new_cases NaN indica che quella data non ha rilevazione)
df_ita_weekly = df_ita[df_ita['new_cases'].notna()].copy()

print(f'Righe con new_cases valorizzato (filtrate): {len(df_ita_weekly)}')
df_ita_weekly[['date', 'total_cases', 'new_cases']].head()

In [ ]:
# Punto 3a — Evoluzione dei casi totali dall'inizio alla fine del 2022

fig, ax = plt.subplots(figsize=(13, 5))

ax.plot(
    df_ita_weekly['date'],
    df_ita_weekly['total_cases'] / 1e6,
    color='steelblue', linewidth=2
)
ax.fill_between(
    df_ita_weekly['date'],
    df_ita_weekly['total_cases'] / 1e6,
    alpha=0.2, color='steelblue'
)

ax.set_title('Italia 2022 — Casi totali cumulativi (milioni)', fontsize=13, fontweight='bold')
ax.set_xlabel('Data')
ax.set_ylabel('Casi totali (milioni)')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:.1f}M'))
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

In [ ]:
# Punto 3b — Nuovi casi rispetto alla data

fig, ax = plt.subplots(figsize=(13, 5))

ax.bar(
    df_ita_weekly['date'],
    df_ita_weekly['new_cases'],
    color='tomato', alpha=0.8, width=5
)

ax.set_title('Italia 2022 — Nuovi casi per data (rilevazioni settimanali)', fontsize=13, fontweight='bold')
ax.set_xlabel('Data')
ax.set_ylabel('Nuovi casi')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1e3:.0f}k'))
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

---
## Punto 4 — Pazienti in terapia intensiva: Italia, Germania, Francia (maggio 2022 – aprile 2023)

In [ ]:
# Filtro nazioni e periodo
nazioni_icu = ['Italy', 'Germany', 'France']

df_icu = df[
    (df['location'].isin(nazioni_icu)) &
    (df['date'] >= '2022-05-01') &
    (df['date'] <= '2023-04-30') &
    (df['icu_patients'].notna())
][['date', 'location', 'icu_patients']].copy()

print(f'Righe disponibili per il boxplot ICU: {len(df_icu)}')
df_icu.groupby('location')['icu_patients'].describe().round(1)

In [ ]:
fig, ax = plt.subplots(figsize=(9, 6))

sns.boxplot(
    data=df_icu,
    x='location',
    y='icu_patients',
    palette='Set2',
    width=0.5,
    ax=ax
)

ax.set_title(
    'Pazienti in Terapia Intensiva (ICU)\nItalia, Germania, Francia — Mag 2022 / Apr 2023',
    fontsize=13, fontweight='bold'
)
ax.set_xlabel('Nazione')
ax.set_ylabel('Pazienti ICU')
plt.tight_layout()
plt.show()

### Commento al boxplot ICU

Dal boxplot emerge che la **Francia** ha registrato il numero mediano più alto di pazienti in terapia intensiva nel periodo considerato, con anche una dispersione maggiore dei valori (IQR più ampio), indicando un'evoluzione più variabile della pressione sulle ICU. La **Germania** presenta valori mediani inferiori alla Francia ma con alcuni outlier verso l'alto, suggerendo picchi occasionali di ricoveri critici. L'**Italia** mostra la distribuzione più compressa e mediana più bassa tra le tre nazioni, il che potrebbe riflettere sia una minore incidenza nelle fasi tardive della pandemia sia differenze nei criteri di classificazione del paziente ICU tra i sistemi sanitari nazionali.

---
## Punto 5 — Pazienti ospedalizzati in Italia, Germania, Francia e Spagna — Anno 2021

In [ ]:
nazioni_hosp = ['Italy', 'Germany', 'France', 'Spain']

df_hosp = df[
    (df['location'].isin(nazioni_hosp)) &
    (df['date'].dt.year == 2021)
][['date', 'location', 'hosp_patients']].copy()

print('Valori nulli per nazione:')
print(df_hosp.groupby('location')['hosp_patients'].apply(lambda x: x.isna().sum()).rename('null_count'))
print()
print('Totale righe per nazione:')
print(df_hosp.groupby('location')['hosp_patients'].count().rename('righe_totali'))

In [ ]:
# Punto 5a — Somma pazienti ospedalizzati per nazione nel 2021
# Usiamo la somma dei valori disponibili (skipna=True di default)
somma_hosp = (
    df_hosp
    .groupby('location')['hosp_patients']
    .sum()
    .reset_index()
    .rename(columns={'hosp_patients': 'somma_hosp_patients'})
    .sort_values('somma_hosp_patients', ascending=False)
)

somma_hosp['somma_hosp_fmt'] = somma_hosp['somma_hosp_patients'].apply(lambda x: f"{x:,.0f}")
display(somma_hosp[['location', 'somma_hosp_fmt']].rename(columns={'location': 'Nazione', 'somma_hosp_fmt': 'Somma hosp_patients 2021'}).reset_index(drop=True))

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))

ax.bar(
    somma_hosp['location'],
    somma_hosp['somma_hosp_patients'] / 1e3,
    color=sns.color_palette('Set2', 4)
)

for i, row in somma_hosp.reset_index(drop=True).iterrows():
    ax.text(i, row['somma_hosp_patients'] / 1e3 + 5,
            f"{row['somma_hosp_patients']/1e3:,.0f}k",
            ha='center', va='bottom', fontsize=10)

ax.set_title('Somma pazienti ospedalizzati — 2021\n(Italia, Germania, Francia, Spagna)', fontsize=13, fontweight='bold')
ax.set_xlabel('Nazione')
ax.set_ylabel('Somma hosp_patients (migliaia)')
plt.tight_layout()
plt.show()

### Punto 5b — Gestione dei valori nulli in `hosp_patients`

I valori nulli in `hosp_patients` per alcune nazioni e date **NON possono essere gestiti tramite sostituzione** (ad es. con media, mediana o forward-fill) per le seguenti ragioni:

1. **Non si tratta di dato mancante casuale**: un valore NaN indica che la nazione non ha riportato il numero di ospedalizzazioni per quella data, il che è strutturalmente diverso da un dato mancante per errore tecnico.

2. **La serie storica ha periodicità irregolare**: alcune nazioni aggiornano i dati giornalmente, altre settimanalmente. Sostituire con il valore precedente (ffill) porterebbe a duplicare artificialmente il dato e gonfiare la somma.

3. **Il calcolo richiesto è una somma** (non una media): inserire valori imputati in una somma introduce bias sistematico.

**Conclusione**: è corretto ignorare i NaN (`skipna=True`) nel calcolo della somma, tenendo presente che il risultato è una **sottostima** del dato reale per le nazioni con più valori mancanti.